In [ ]:
import kagglehub
path = kagglehub.dataset_download("omkargurav/face-mask-dataset")
print("Path to dataset files:", path)

In [ ]:
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [ ]:
IMG_SIZE  = (224, 224)   # MobileNetV2 expects 224x224
BATCH     = 32
DATA_DIR  = path + "/data"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="binary", subset="training", shuffle=True, seed=42
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="binary", subset="validation", shuffle=False, seed=42
)

print("Classes:", train_gen.class_indices)
print("Train batches:", len(train_gen))
print("Val   batches:", len(val_gen))

In [ ]:
# Load MobileNetV2 - pretrained on ImageNet, top layer hatao
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,         # Last classification layer nahi chahiye
    weights="imagenet"         # Pretrained weights use karo
)

# Phase 1: Base model freeze karo (sirf top layers train honge)
base_model.trainable = False

# Custom head add karo
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Phase 1 Training - Sirf top layers
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('mask_mobilenet_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print("Phase 1: Training top layers...")
history1 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks
)

print("Phase 1 complete!")

In [ ]:
# Phase 2: Fine-tuning - Last 30 layers unfreeze karo
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Lower learning rate for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable layers: {sum(1 for l in model.layers if l.trainable)}")

print("Phase 2: Fine-tuning...")
history2 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks
)

model.save("mask_mobilenet_final.h5")
print("Model saved!")

In [ ]:
import matplotlib.pyplot as plt

# Combine both phase histories
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss    = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']

epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Train Accuracy')
plt.plot(epochs, val_acc, label='Val Accuracy')
plt.axvline(x=len(history1.history['accuracy']), color='r', linestyle='--', label='Fine-tune start')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Val Loss')
plt.axvline(x=len(history1.history['loss']), color='r', linestyle='--', label='Fine-tune start')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('training_plot.png')
plt.show()